# 2AFC Constant-Stimuli Psychophysics Analysis

This notebook analyzes a two-alternative forced-choice (2AFC) constant-stimuli experiment, following the relevant perception-analysis structure used by Farajian & Nisky: count matrices of comparison-standard stiffness differences, logistic psychometric curves, PSE, and JND.

**Raw data rule:** set `DATA_ROOT` to the main data folder. The notebook treats it as read-only and writes all outputs to `OUTPUT_ROOT`.

**Important fitting rule:** the raw `answer` column is **not** used directly for fitting. Each trial is first converted into `response_comparison_greater` after determining whether the comparison stimulus was object 1 or object 2.

**Added for this experiment:** success/correctness is analyzed separately from the psychometric response, including answer duration (`time_to_answer`) and fatigue/learning proxies (trial order, elapsed time, first-vs-second half, and early/middle/late bins) per participant and across participants.


## 0. Configuration

Edit `DATA_ROOT`. Keep `OUTPUT_ROOT` outside the raw data folder.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Works whether Jupyter's current directory is the project root, analysis/, or this folder.
CWD = Path.cwd()
if (CWD / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD
elif (CWD / "analysis" / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "analysis" / "psychophysics_analysis"
elif (CWD / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "psychophysics_analysis"
else:
    raise FileNotFoundError("Could not locate twoafc_psychophysics.py. Open the notebook from the project root, analysis/, or analysis/psychophysics_analysis/.")
sys.path.insert(0, str(ANALYSIS_DIR))
import twoafc_psychophysics as pf

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Main read-only results folder: P* are pilots, E* are experiment participants.
DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\expiriments_results")

# Separate output location. Do not place this inside DATA_ROOT.
OUTPUT_ROOT = ANALYSIS_DIR / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STANDARD_FALLBACK = 85.0          # used only as fallback/tie-break anchor
STANDARD_ABS_TOLERANCE = 0.75     # tolerance for classifying the inferred standard
FIG_DPI = 160

# Optional manual column overrides if automatic detection needs help.
# Example: {"object_1_value": "object_1_stiffness", "answer": "answer"}
MANUAL_COLUMN_OVERRIDES = {}

print("ANALYSIS_DIR:", ANALYSIS_DIR.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
if str(DATA_ROOT) == "PUT_MAIN_DATA_FOLDER_HERE":
    print("?? Set DATA_ROOT before running the analysis cells.")


ANALYSIS_DIR: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics_analysis
DATA_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\expiriments_results
OUTPUT_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics_analysis\results


## 1. File discovery and loading

One folder per subject is expected under `DATA_ROOT`. The code searches each subject folder recursively for files named exactly `answers.csv` only, then saves a discovery audit table.

In [2]:
pf.validate_paths(DATA_ROOT, OUTPUT_ROOT)

file_discovery_summary = pf.discover_answer_files(DATA_ROOT, OUTPUT_ROOT)
pf.save_csv(file_discovery_summary, OUTPUT_ROOT, "file_discovery_summary.csv")
display(file_discovery_summary.head(30))

combined_raw_imported_data = pf.load_selected_subject_csvs(file_discovery_summary)
pf.save_csv(combined_raw_imported_data, OUTPUT_ROOT, "combined_raw_imported_data.csv")
display(combined_raw_imported_data.head())
print("combined raw shape:", combined_raw_imported_data.shape)

,subject_id,subject_folder,candidate_rank,candidate_score,selected,source_file,source_file_name,n_answer_csv_candidates,n_total_csv_files_in_subject_folder,selection_warning
0,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,260,
1,E10,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
2,E11,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,274,
3,E12,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,273,
4,E2,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,266,
5,E3,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,274,
6,E4,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,272,
7,E5,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,276,
8,E6,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,273,
9,E7,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,273,


,subject_id,_subject_folder,_source_file,_source_file_name,_row_in_source,timestamp,pair_number,object_1_finger,object_1_stiffness,object_2_finger,object_2_stiffness,time_to_answer,answer
0,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,0,2026-05-09T16:42:37.905766,1,pinky,125,pinky,85,5.438098,0
1,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,2026-05-09T16:43:09.191703,2,pinky,45,pinky,85,3.965589,0
2,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,2,2026-05-09T16:43:41.424086,3,pinky,25,pinky,85,2.604255,1
3,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,3,2026-05-09T16:44:29.923436,4,pinky,165,pinky,85,2.336992,0
4,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,4,2026-05-09T16:44:55.500089,5,pinky,85,pinky,145,2.118856,1


combined raw shape: (3448, 13)


## 2. Column detection and validation

Detects object values, object fingers, answer, reaction time, trial index, timestamp, and optional block columns. Review the detection table; if needed, set `MANUAL_COLUMN_OVERRIDES` above and rerun.

In [3]:
detected_columns, column_detection_details = pf.detect_columns(combined_raw_imported_data, MANUAL_COLUMN_OVERRIDES)
print(json.dumps(detected_columns, indent=2, ensure_ascii=False))
pf.save_csv(column_detection_details, OUTPUT_ROOT, "column_detection_details.csv")
display(column_detection_details.groupby("target").head(5))

missing_required = [k for k in ["object_1_value", "object_2_value", "answer"] if not detected_columns.get(k)]
if missing_required:
    raise RuntimeError(f"Missing required detected columns: {missing_required}. Add MANUAL_COLUMN_OVERRIDES and rerun.")

{
  "object_1_value": "object_1_stiffness",
  "object_2_value": "object_2_stiffness",
  "object_1_finger": "object_1_finger",
  "object_2_finger": "object_2_finger",
  "answer": "answer",
  "reaction_time": "time_to_answer",
  "trial_index": "pair_number",
  "timestamp": "timestamp",
  "block": null
}


,target,column,normalized,score,manual
0,object_1_value,object_1_stiffness,object_1_stiffness,23,False
1,object_1_value,object_2_stiffness,object_2_stiffness,9,False
2,object_1_value,object_1_finger,object_1_finger,6,False
3,object_1_value,_row_in_source,row_in_source,0,False
4,object_1_value,_source_file,source_file,0,False
8,object_2_value,object_2_stiffness,object_2_stiffness,23,False
9,object_2_value,object_1_stiffness,object_1_stiffness,9,False
10,object_2_value,object_2_finger,object_2_finger,6,False
11,object_2_value,_row_in_source,row_in_source,0,False
12,object_2_value,_source_file,source_file,0,False


## 3. Trial cleaning and canonicalization

Protocol rows and invalid/ambiguous trials are flagged and saved separately. Valid trials receive `response_comparison_greater` for fitting.

Examples:
- standard object 1, comparison object 2 ? response is 1 when `answer == 1`.
- standard object 2, comparison object 1 ? response is 1 when `answer == 0`.

In [4]:
inferred_standard_value, standard_inference_table = pf.infer_standard_value(combined_raw_imported_data, detected_columns, STANDARD_FALLBACK)
print(f"Inferred standard/reference value: {inferred_standard_value:g}")
pf.save_csv(standard_inference_table, OUTPUT_ROOT, "standard_value_inference.csv")
display(standard_inference_table.head(10))

combined_clean_trials, combined_flagged_trials = pf.canonicalize_trials(
    combined_raw_imported_data,
    detected_columns,
    standard_value=inferred_standard_value,
    standard_tolerance=STANDARD_ABS_TOLERANCE,
)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(combined_flagged_trials, OUTPUT_ROOT, "combined_flagged_trials.csv")

print("Clean trials:", combined_clean_trials.shape)
display(combined_clean_trials.head())
print("Flagged trials:", combined_flagged_trials.shape)
display(combined_flagged_trials.head())

Inferred standard/reference value: 85


,candidate_value,count,n_unique_partners,distance_from_fallback,standard_score
0,85,3448,14,0.0,1.350000
1,70,360,1,15.0,0.120033
2,100,320,1,15.0,0.108432
3,25,416,1,60.0,0.108150
4,145,416,1,60.0,0.108150
5,130,360,1,45.0,0.101283
6,40,360,1,45.0,0.101283
7,115,320,1,30.0,0.099057
8,55,320,1,30.0,0.099057
9,65,96,1,20.0,0.040342


Clean trials: (3448, 30)


,subject_id,source_file,source_file_name,row_in_source,trial_index_raw,timestamp,block_raw,object_1_value,object_2_value,object_1_finger_raw,object_2_finger_raw,object_1_finger,object_2_finger,standard_value,standard_side,comparison_side,comparison_value,standard_finger,comparison_finger,finger_condition,raw_answer,answer_code,response_comparison_greater,reaction_time,protocol_marker,finger_warning,excluded_from_fit,flag_reason,global_trial_order,answer_chose_object_2
0,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,0,1,2026-05-09T16:42:37.905766,NaN,125,85,pinky,pinky,P,P,85.0,object_2,object_1,125.0,P,P,P,0,0,1,5.438098,False,,False,,1,0.0
1,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,2,2026-05-09T16:43:09.191703,NaN,45,85,pinky,pinky,P,P,85.0,object_2,object_1,45.0,P,P,P,0,0,1,3.965589,False,,False,,2,0.0
2,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,2,3,2026-05-09T16:43:41.424086,NaN,25,85,pinky,pinky,P,P,85.0,object_2,object_1,25.0,P,P,P,1,1,0,2.604255,False,,False,,3,1.0
3,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,3,4,2026-05-09T16:44:29.923436,NaN,165,85,pinky,pinky,P,P,85.0,object_2,object_1,165.0,P,P,P,0,0,1,2.336992,False,,False,,4,0.0
4,E1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,4,5,2026-05-09T16:44:55.500089,NaN,85,145,pinky,pinky,P,P,85.0,object_1,object_2,145.0,P,P,P,1,1,1,2.118856,False,,False,,5,1.0


Flagged trials: (0, 0)


""


## 4. Block and finger detection

Finger labels are normalized to `I`, `M`, `R`, `P`. Block order is inferred from changes in `finger_condition` within each subject.

In [5]:
combined_clean_trials, block_order_summary = pf.add_block_numbers(combined_clean_trials)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(block_order_summary, OUTPUT_ROOT, "block_order_summary.csv")
display(block_order_summary.head(30))
display(pd.crosstab(combined_clean_trials["subject_id"], combined_clean_trials["finger_condition"]))

,subject_id,block_number,finger_condition,first_global_trial_order,n_trials_in_block
0,E1,1,P,1,64
1,E1,2,I,65,64
2,E1,3,R,129,64
3,E1,4,M,193,64
4,E10,1,I,1,3
5,E10,2,M,4,3
6,E10,3,R,7,3
7,E10,4,P,10,3
8,E10,5,R,13,64
9,E10,6,P,77,64


finger_condition,I,M,P,R
subject_id,,,,
E1,64,64,64,64
E10,67,67,67,67
E11,67,67,67,67
E12,67,67,67,67
E2,64,64,64,64
E3,67,67,67,67
E4,67,67,67,67
E5,67,67,67,67
E6,67,67,67,67


## 5. Success/correctness coding and Farajian-style count matrices

Psychometric fitting uses `response_comparison_greater`. Success-rate analysis asks a different question: did the participant choose the physically stiffer stimulus? This cell adds `correct_response`, signed stiffness deltas, elapsed time, participant group (`pilot`/`experiment`), and fatigue/order proxies.


In [6]:
combined_success_trials = pf.add_success_and_time_columns(combined_clean_trials)
pf.save_csv(combined_success_trials, OUTPUT_ROOT, "combined_success_trials.csv")

farajian_style_input_by_subject_finger = pf.make_farajian_style_psychometric_input(combined_success_trials, ["subject_id", "finger_condition"])
farajian_style_input_group_by_finger = pf.make_farajian_style_psychometric_input(combined_success_trials, ["finger_condition"])
pf.save_csv(farajian_style_input_by_subject_finger, OUTPUT_ROOT, "farajian_style_input_by_subject_finger.csv")
pf.save_csv(farajian_style_input_group_by_finger, OUTPUT_ROOT, "farajian_style_input_group_by_finger.csv")

display(combined_success_trials[[
    "subject_id", "subject_group_label", "finger_condition", "global_trial_order",
    "signed_stiffness_delta", "response_comparison_greater", "correct_response",
    "reaction_time", "elapsed_minutes", "session_half", "fatigue_tertile"
]].head(20))
display(farajian_style_input_by_subject_finger.head(30))


,subject_id,subject_group_label,finger_condition,global_trial_order,signed_stiffness_delta,response_comparison_greater,correct_response,reaction_time,elapsed_minutes,session_half,fatigue_tertile
0,E1,experiment,P,1,40.0,1,1.0,5.438098,0.000000,first_half,early
1,E1,experiment,P,2,-40.0,1,0.0,3.965589,0.521432,first_half,early
2,E1,experiment,P,3,-60.0,0,1.0,2.604255,1.058639,first_half,early
3,E1,experiment,P,4,80.0,1,1.0,2.336992,1.866961,first_half,early
4,E1,experiment,P,5,60.0,1,1.0,2.118856,2.293239,first_half,early
5,E1,experiment,P,6,-20.0,0,1.0,2.327784,2.727673,first_half,early
6,E1,experiment,P,7,20.0,0,0.0,2.093581,3.195846,first_half,early
7,E1,experiment,P,8,-80.0,0,1.0,2.462920,3.594276,first_half,early
8,E1,experiment,P,9,-20.0,1,0.0,2.181055,3.914480,first_half,early
9,E1,experiment,P,10,-40.0,0,1.0,1.959361,4.133198,first_half,early


,subject_id,finger_condition,delta_comparison_minus_standard,delta_standard_minus_comparison,n_comparison_greater,n_trials,p_comparison_greater,n_correct,success_rate,mean_reaction_time,standard_value,comparison_value,signed_stiffness_delta
0,E1,I,-80.0,80.0,0,8,0.000,8.0,1.000,1.926551,85.0,5.0,-80.0
1,E1,I,-60.0,60.0,0,8,0.000,8.0,1.000,1.916106,85.0,25.0,-60.0
2,E1,I,-40.0,40.0,0,8,0.000,8.0,1.000,1.904801,85.0,45.0,-40.0
3,E1,I,-20.0,20.0,3,8,0.375,5.0,0.625,2.360614,85.0,65.0,-20.0
4,E1,I,20.0,-20.0,7,8,0.875,7.0,0.875,2.116333,85.0,105.0,20.0
5,E1,I,40.0,-40.0,8,8,1.000,8.0,1.000,2.122745,85.0,125.0,40.0
6,E1,I,60.0,-60.0,8,8,1.000,8.0,1.000,1.941267,85.0,145.0,60.0
7,E1,I,80.0,-80.0,8,8,1.000,8.0,1.000,1.909946,85.0,165.0,80.0
8,E1,M,-80.0,80.0,0,8,0.000,8.0,1.000,2.078783,85.0,5.0,-80.0
9,E1,M,-60.0,60.0,0,8,0.000,8.0,1.000,1.923015,85.0,25.0,-60.0


## 6. Quality-control summaries

Flags suspicious cases without silently deleting more data: missing/invalid rows, trial counts per value/finger, non-monotonic curves, apparent error rate, RT outliers, side bias, standard-side bias, and related warnings.

In [7]:
qc_summary = pf.make_qc_summary(combined_clean_trials, combined_flagged_trials)
pf.save_csv(qc_summary, OUTPUT_ROOT, "qc_summary.csv")
display(qc_summary)

,subject_id,finger_condition,n_clean_trials,n_stimulus_levels,min_trials_per_value,median_trials_per_value,side_bias_p_chose_object2,standard_side_p_object2,apparent_error_rate,rt_outlier_count,monotonic_violations,spearman_r,non_monotonic_flag,qc_warnings,n_flagged_rows
0,E1,I,64,8,8,8.0,0.500000,0.500000,0.062500,2,0,0.951190,False,,0
1,E1,M,64,8,8,8.0,0.468750,0.500000,0.093750,5,0,0.969782,False,,0
2,E1,P,64,8,8,8.0,0.484375,0.500000,0.140625,7,0,0.981981,False,,0
3,E1,R,64,8,8,8.0,0.484375,0.500000,0.046875,5,0,0.932227,False,,0
4,E10,I,67,8,8,8.0,0.432836,0.492537,0.134581,3,0,0.970077,False,,0
5,E10,M,67,8,8,8.0,0.477612,0.507463,0.104278,2,0,0.988024,False,,0
6,E10,P,67,8,8,8.0,0.432836,0.477612,0.089127,4,0,0.951876,False,,0
7,E10,R,67,8,8,8.0,0.537313,0.507463,0.104724,6,0,0.951876,False,,0
8,E11,I,67,8,8,8.0,0.477612,0.492537,0.180481,6,0,0.963925,False,,0
9,E11,M,67,8,8,8.0,0.477612,0.507463,0.135918,4,1,0.951503,True,non_monotonic_curve,0


## 7. Psychometric aggregation

Aggregates valid trials into binomial counts: `comparison_value`, `n_trials`, `n_comparison_greater`, and observed probability.

In [8]:
psychometric_input_by_subject_finger = pf.make_psychometric_input(combined_clean_trials, ["subject_id", "finger_condition"])
pf.save_csv(psychometric_input_by_subject_finger, OUTPUT_ROOT, "psychometric_input_by_subject_finger.csv")
display(psychometric_input_by_subject_finger.head(30))

,subject_id,finger_condition,comparison_value,n_trials,n_comparison_greater,p_comparison_greater,mean_rt,standard_value
0,E1,I,5.0,8,0,0.000,1.926551,85.0
1,E1,I,25.0,8,0,0.000,1.916106,85.0
2,E1,I,45.0,8,0,0.000,1.904801,85.0
3,E1,I,65.0,8,3,0.375,2.360614,85.0
4,E1,I,105.0,8,7,0.875,2.116333,85.0
5,E1,I,125.0,8,8,1.000,2.122745,85.0
6,E1,I,145.0,8,8,1.000,1.941267,85.0
7,E1,I,165.0,8,8,1.000,1.909946,85.0
8,E1,M,5.0,8,0,0.000,2.078783,85.0
9,E1,M,25.0,8,0,0.000,1.923015,85.0


## Psychometric fitting setup

The notebook attempts Wichmann-lab `psignifit` first when available and compatible. If not, it prints installation guidance and uses the scipy logistic fallback on physical stimulus values.

In [9]:
PSIGNIFIT_AVAILABLE, PSIGNIFIT_STATUS = pf.check_psignifit_available()
print(PSIGNIFIT_STATUS)
if not PSIGNIFIT_AVAILABLE:
    print("Optional psignifit install example: pip install psignifit")
    print("Continuing with scipy logistic fallback fits.")

psignifit import succeeded (version=unknown)


## 8. Subject x finger psychometric fits

In [10]:
pse_jnd_by_subject_finger = pf.fit_conditions(psychometric_input_by_subject_finger, ["subject_id", "finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(pse_jnd_by_subject_finger, OUTPUT_ROOT, "pse_jnd_by_subject_finger.csv")
display(pse_jnd_by_subject_finger)

,subject_id,finger_condition,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,E1,I,scipy_logistic_fallback,76.134600,11.955487,64.179112,88.090087,0.022973,0.000000,0.000000,0.000000,ok,,64,8,2.103563,26.716898,psignifit_failed: 'dict' object has no attribu...,9.358449,76.134600,10.882353,5.0,165.0
1,E1,M,scipy_logistic_fallback,100.433106,14.025326,86.407781,114.458432,0.019583,0.000000,0.000000,0.000000,ok,,64,8,1.585911,32.732911,psignifit_failed: 'dict' object has no attribu...,12.366456,100.433106,12.766402,5.0,165.0
2,E1,P,scipy_logistic_fallback,95.744883,23.034181,74.670052,120.738415,0.012402,0.103729,0.000000,0.103729,ok,,64,8,1.065465,52.761129,psignifit_failed: 'dict' object has no attribu...,22.380565,91.600232,17.825773,5.0,165.0
3,E1,R,scipy_logistic_fallback,98.969443,9.999349,88.970095,108.968792,0.027467,0.000000,0.000000,0.000000,ok,,64,8,1.426814,24.452499,psignifit_failed: 'dict' object has no attribu...,8.226250,98.969443,9.101800,5.0,165.0
4,E10,I,scipy_logistic_fallback,85.485339,18.482478,67.692312,104.657267,0.015084,0.053352,0.000000,0.053352,ok,,67,8,3.577245,57.442304,psignifit_failed: 'dict' object has no attribu...,24.721152,83.720544,15.640064,25.0,145.0
5,E10,M,scipy_logistic_fallback,76.408379,10.956600,66.048269,87.961470,0.025650,0.073351,0.000000,0.073351,ok,,67,8,2.023803,45.845003,psignifit_failed: 'dict' object has no attribu...,18.922502,74.984541,8.974936,25.0,145.0
6,E10,P,scipy_logistic_fallback,76.111447,10.284597,66.118064,86.687257,0.027001,0.041839,0.000000,0.041839,ok,,67,8,2.805090,40.597968,psignifit_failed: 'dict' object has no attribu...,16.298984,75.337657,8.854687,25.0,145.0
7,E10,R,scipy_logistic_fallback,91.198310,12.626983,78.242692,103.496657,0.021971,0.038783,0.038783,0.000000,ok,,67,8,3.571906,43.998930,psignifit_failed: 'dict' object has no attribu...,17.999465,92.079956,10.919673,25.0,145.0
8,E11,I,scipy_logistic_fallback,100.143312,22.519514,77.623798,122.662826,0.012196,0.000000,0.000000,0.000000,ok,,67,8,6.568184,62.471803,psignifit_failed: 'dict' object has no attribu...,27.235901,100.143312,20.498145,25.0,145.0
9,E11,M,scipy_logistic_fallback,102.464535,16.626594,85.837940,119.091129,0.016519,0.000000,0.000000,0.000000,ok,,67,8,6.581874,50.483281,psignifit_failed: 'dict' object has no attribu...,21.241640,102.464535,15.134178,25.0,145.0


## 9. Subject-level pooled-across-fingers fits

In [11]:
psychometric_input_subject_pooled = pf.make_psychometric_input(combined_clean_trials, ["subject_id"])
pse_jnd_subject_pooled = pf.fit_conditions(psychometric_input_subject_pooled, ["subject_id"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_subject_pooled, OUTPUT_ROOT, "psychometric_input_subject_pooled.csv")
pf.save_csv(pse_jnd_subject_pooled, OUTPUT_ROOT, "pse_jnd_subject_pooled.csv")
display(pse_jnd_subject_pooled)

,subject_id,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,E1,scipy_logistic_fallback,92.751105,16.133410,76.834679,109.101499,0.017109,0.021017,0.000000,0.021017,ok,,256,8,1.196932,127.496952,psignifit_failed: 'dict' object has no attribu...,59.748476,92.137074,14.298771,5.0,165.0
1,E10,scipy_logistic_fallback,83.359156,15.124147,68.506649,98.754943,0.018283,0.027586,0.000000,0.027586,ok,,268,8,1.351860,169.335032,psignifit_failed: 'dict' object has no attribu...,80.667516,82.605130,13.286198,25.0,145.0
2,E11,scipy_logistic_fallback,91.276812,19.734667,73.033640,112.502975,0.014398,0.094971,0.000000,0.094971,ok,,268,8,13.365992,212.697433,psignifit_failed: 'dict' object has no attribu...,102.348716,88.003108,15.541081,25.0,145.0
3,E12,scipy_logistic_fallback,86.310487,30.259179,58.050249,118.568607,0.009411,0.108884,0.014388,0.094496,ok,,268,8,4.314183,280.130583,psignifit_failed: 'dict' object has no attribu...,136.065292,82.077221,23.481753,25.0,145.0
4,E2,scipy_logistic_fallback,93.505649,26.300164,73.959656,126.559985,0.012464,0.200000,0.000000,0.200000,warning,high_lapse_estimate,256,8,4.484356,202.255295,psignifit_failed: 'dict' object has no attribu...,97.127648,85.820954,15.043676,5.0,165.0
5,E3,scipy_logistic_fallback,88.362440,20.665340,72.425479,113.756159,0.015890,0.243115,0.043115,0.200000,warning,high_lapse_estimate,268,8,8.902961,251.547259,psignifit_failed: 'dict' object has no attribu...,121.773630,83.568484,11.396541,25.0,145.0
6,E4,scipy_logistic_fallback,79.159069,55.787617,37.698326,149.273560,0.005876,0.200000,0.000000,0.200000,warning,high_lapse_estimate,268,8,8.698299,333.788509,psignifit_failed: 'dict' object has no attribu...,162.894255,62.858378,31.910480,25.0,145.0
7,E5,scipy_logistic_fallback,82.496303,12.152137,71.610148,95.914423,0.023881,0.132312,0.008429,0.123883,ok,,268,8,3.505009,180.824410,psignifit_failed: 'dict' object has no attribu...,86.412205,80.107709,8.922492,25.0,145.0
8,E6,scipy_logistic_fallback,85.019546,16.254613,71.496096,104.005321,0.018689,0.173628,0.008859,0.164769,warning,high_lapse_estimate,268,8,15.507435,207.759848,psignifit_failed: 'dict' object has no attribu...,99.879924,80.948101,10.660671,25.0,145.0
9,E7,scipy_logistic_fallback,75.125729,16.649231,62.752203,96.050666,0.019688,0.200000,0.000000,0.200000,warning,high_lapse_estimate,268,8,3.843211,232.801115,psignifit_failed: 'dict' object has no attribu...,112.400558,70.260958,9.523349,25.0,145.0


## 10. Group-level fits by finger

Pools all valid trials across subjects within each finger condition.

In [12]:
psychometric_input_group_by_finger = pf.make_psychometric_input(combined_clean_trials, ["finger_condition"])
pse_jnd_group_by_finger = pf.fit_conditions(psychometric_input_group_by_finger, ["finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_by_finger, OUTPUT_ROOT, "psychometric_input_group_by_finger.csv")
pf.save_csv(pse_jnd_group_by_finger, OUTPUT_ROOT, "pse_jnd_group_by_finger.csv")
display(pse_jnd_group_by_finger)

,finger_condition,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,I,scipy_logistic_fallback,89.936426,24.475073,67.922611,116.872758,0.011959,0.157683,0.027241,0.130442,ok,,862,14,8.955702,775.650718,psignifit_failed: 'dict' object has no attribu...,383.825359,85.664768,17.344725,5.0,165.0
1,M,scipy_logistic_fallback,87.625573,21.529539,68.015256,111.074334,0.013304,0.108315,0.000642,0.107672,ok,,862,14,15.802793,696.220214,psignifit_failed: 'dict' object has no attribu...,344.110107,83.641777,16.514843,5.0,165.0
2,P,scipy_logistic_fallback,86.915452,19.038980,68.904651,106.982612,0.014758,0.072870,0.000000,0.072870,ok,,862,14,9.789935,631.521441,psignifit_failed: 'dict' object has no attribu...,311.760721,84.456833,15.608335,5.0,165.0
3,R,scipy_logistic_fallback,84.574860,22.138518,65.757432,110.034467,0.013561,0.168587,0.012481,0.156106,warning,high_lapse_estimate,862,14,22.560353,743.415631,psignifit_failed: 'dict' object has no attribu...,367.707815,79.385262,14.870151,5.0,165.0


## 11. Group-level all-pooled fit

In [13]:
all_pooled_trials = combined_clean_trials.copy()
all_pooled_trials["group"] = "all_pooled"
psychometric_input_group_all_pooled = pf.make_psychometric_input(all_pooled_trials, ["group"])
pse_jnd_group_all_pooled = pf.fit_conditions(psychometric_input_group_all_pooled, ["group"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_all_pooled, OUTPUT_ROOT, "psychometric_input_group_all_pooled.csv")
pf.save_csv(pse_jnd_group_all_pooled, OUTPUT_ROOT, "pse_jnd_group_all_pooled.csv")
display(pse_jnd_group_all_pooled)

,group,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,all_pooled,scipy_logistic_fallback,87.233766,22.092733,67.229083,111.414549,0.013026,0.117492,0.003681,0.113811,ok,,3448,14,17.693769,2836.6474,psignifit_failed: 'dict' object has no attribu...,1414.3237,83.050418,16.6739,5.0,165.0


## 12. Subject-average group analysis

Computes subject-level probabilities first, then averages subjects so each subject has equal weight. This complements trial-pooled group fits.

In [14]:
subject_average_group_by_finger = pf.subject_average_psychometric(combined_clean_trials, ["finger_condition"])
subject_average_group_all = pf.subject_average_psychometric(combined_clean_trials.assign(group="all_subject_average"), ["group"])
pf.save_csv(subject_average_group_by_finger, OUTPUT_ROOT, "subject_average_group_by_finger.csv")
pf.save_csv(subject_average_group_all, OUTPUT_ROOT, "subject_average_group_all_pooled.csv")
display(subject_average_group_by_finger.head(30))

,finger_condition,comparison_value,n_subjects,mean_p_comparison_greater,sem_p_comparison_greater,total_trials,standard_value
0,I,5.0,3,0.041667,0.041667,24,85.0
1,I,25.0,13,0.038462,0.021856,104,85.0
2,I,40.0,10,0.111111,0.054935,90,85.0
3,I,45.0,3,0.083333,0.041667,24,85.0
4,I,55.0,10,0.150000,0.040825,80,85.0
5,I,65.0,3,0.333333,0.110240,24,85.0
6,I,70.0,10,0.222222,0.040572,90,85.0
7,I,100.0,10,0.625000,0.052705,80,85.0
8,I,105.0,3,0.750000,0.072169,24,85.0
9,I,115.0,10,0.687500,0.027951,80,85.0


## 13. Order-effect analysis

Summarizes fatigue/order effects across trial number and inferred block order.

In [15]:
order_effects_summary, order_effects_binned = pf.compute_order_effects(combined_clean_trials)
pf.save_csv(order_effects_summary, OUTPUT_ROOT, "order_effects_summary.csv")
pf.save_csv(order_effects_binned, OUTPUT_ROOT, "order_effects_binned.csv")
display(order_effects_summary)

,subject_id,finger_condition,n_trials,first_half_p_comparison_greater,second_half_p_comparison_greater,first_half_mean_rt,second_half_mean_rt,spearman_response_vs_order,spearman_rt_vs_order,first_block_number
0,E1,I,64,0.531250,0.531250,2.088419,1.961172,0.022035,-0.237179,2
1,E1,M,64,0.437500,0.437500,2.200159,1.805794,-0.037511,-0.448993,4
2,E1,P,64,0.437500,0.406250,2.491971,2.393507,-0.067652,-0.292720,1
3,E1,R,64,0.437500,0.468750,1.952724,1.945272,-0.000850,-0.098077,3
4,E10,I,67,0.575758,0.382353,2.475518,2.455543,-0.211678,-0.028693,1
5,E10,M,67,0.545455,0.470588,2.792674,2.150120,-0.067924,-0.455982,2
6,E10,P,67,0.484848,0.558824,2.464267,2.213386,0.041718,-0.249820,4
7,E10,R,67,0.454545,0.500000,7.047507,2.304028,0.049443,-0.413800,3
8,E11,I,67,0.424242,0.382353,2.227676,2.010826,-0.055071,-0.038151,1
9,E11,M,67,0.393939,0.382353,2.413769,2.231409,-0.031675,-0.116450,2


## 14. Success-rate, answer-duration, and fatigue/learning analysis

This section tests the requested psychophysics-only questions:

- Does answer duration (`time_to_answer`) relate to success?
- Does success go up or down over the session (learning vs tiredness/fatigue proxy)?
- Are the effects visible within each participant/finger and across participants?

Because no subjective tiredness scale is present in `answers.csv`, tiredness is operationalized conservatively as trial order, elapsed time, and first-vs-second-half changes.


In [16]:
success_time_fatigue = pf.compute_success_time_fatigue(combined_success_trials, ["subject_id", "finger_condition"])

success_summary_by_subject_finger = success_time_fatigue["summary"]
success_trend_slopes_by_subject_finger = success_time_fatigue["slopes"]
success_by_reaction_time_bin = success_time_fatigue["reaction_time_bins"]
success_by_order_bin = success_time_fatigue["order_bins"]
fatigue_first_second_summary = success_time_fatigue["first_second"]
between_subject_success_time_stats = success_time_fatigue["between_subjects"]
success_summary_by_subject = success_time_fatigue["subject_summary"]

pf.save_csv(success_summary_by_subject, OUTPUT_ROOT, "success_summary_by_subject.csv")
pf.save_csv(success_summary_by_subject_finger, OUTPUT_ROOT, "success_summary_by_subject_finger.csv")
pf.save_csv(success_trend_slopes_by_subject_finger, OUTPUT_ROOT, "success_trend_slopes_by_subject_finger.csv")
pf.save_csv(success_by_reaction_time_bin, OUTPUT_ROOT, "success_by_reaction_time_bin.csv")
pf.save_csv(success_by_order_bin, OUTPUT_ROOT, "success_by_order_bin.csv")
pf.save_csv(fatigue_first_second_summary, OUTPUT_ROOT, "fatigue_first_second_summary.csv")
pf.save_csv(between_subject_success_time_stats, OUTPUT_ROOT, "between_subject_success_time_stats.csv")

print("Participant-level success summary")
display(success_summary_by_subject)
print("Within participant/finger success trend slopes: positive=success improves, negative=success decreases")
display(success_trend_slopes_by_subject_finger)
print("First vs second half fatigue/learning proxy")
display(fatigue_first_second_summary)
print("Between-participant duration/time statistics")
display(between_subject_success_time_stats)


Participant-level success summary


,subject_id,subject_group_label,n_trials,success_rate,mean_reaction_time,median_reaction_time,session_duration_min,mean_abs_stiffness_delta
0,E1,experiment,256,0.914062,2.104877,1.890275,66.549613,50.00000
1,E10,experiment,268,0.891791,2.977327,2.107130,84.758794,37.38806
2,E11,experiment,268,0.858209,2.148858,1.915955,74.676195,37.38806
3,E12,experiment,268,0.779851,2.417485,2.131556,74.118833,37.38806
4,E2,experiment,256,0.828125,2.127112,1.947843,87.672255,50.00000
5,E3,experiment,268,0.813433,2.257843,1.869651,88.423661,37.38806
6,E4,experiment,268,0.682836,2.288773,2.165501,58.798110,37.38806
7,E5,experiment,268,0.891791,2.279084,2.030381,58.605595,37.38806
8,E6,experiment,268,0.847015,2.008074,1.890265,58.608434,37.38806
9,E7,experiment,268,0.817164,1.946395,1.869485,67.107354,37.38806


Within participant/finger success trend slopes: positive=success improves, negative=success decreases


,subject_id,finger_condition,n_trials,success_rate,success_vs_order_slope,success_vs_order_p_value,success_vs_order_spearman_r,success_vs_order_spearman_p,success_vs_order_direction,success_vs_global_order_slope,success_vs_global_order_p_value,success_vs_global_order_spearman_r,success_vs_global_order_spearman_p,success_vs_global_order_direction,success_vs_elapsed_slope,success_vs_elapsed_p_value,success_vs_elapsed_direction,success_vs_reaction_time_slope,success_vs_reaction_time_p_value,success_vs_reaction_time_spearman_r,success_vs_reaction_time_spearman_p,success_vs_reaction_time_direction
0,E1,I,64,0.937500,0.049038,0.641023,0.059403,0.641023,up_weak,0.198489,0.641023,0.059403,0.641023,up_weak,0.325425,0.531636,up_weak,-0.023824,7.214344e-01,-0.122300,3.356800e-01,flat
1,E1,M,64,0.906250,-0.086538,0.493965,-0.087055,0.493965,down_weak,-0.350275,0.493965,-0.087055,0.493965,down_weak,-0.503677,0.363110,down_weak,-0.039882,4.502381e-01,-0.058037,6.487291e-01,down_weak
2,E1,P,64,0.859375,0.206250,0.169189,0.173967,0.169189,up_weak,0.834821,0.169189,0.173967,0.169189,up_weak,0.659488,0.150422,up_weak,-0.007474,8.585132e-01,-0.023115,8.561355e-01,flat
3,E1,R,64,0.953125,0.007212,0.937464,0.010004,0.937464,flat,0.029190,0.937464,0.010004,0.937464,flat,0.104176,0.852754,up_weak,-0.044699,4.047679e-01,-0.066027,6.041987e-01,down_weak
4,E10,I,67,0.865672,0.010536,0.942038,0.009053,0.942038,flat,0.184522,0.398315,0.009053,0.942038,up_weak,0.210988,0.327615,up_weak,-0.223080,1.122121e-08,-0.556755,9.949817e-07,down_p_lt_0.05
5,E10,M,67,0.895522,0.318701,0.011997,0.305299,0.011997,up_p_lt_0.05,0.906174,0.000248,0.305299,0.011997,up_p_lt_0.05,0.889043,0.000251,up_p_lt_0.05,-0.139179,2.147014e-04,-0.370900,2.002714e-03,down_p_lt_0.05
6,E10,P,67,0.910448,0.213345,0.075081,0.218932,0.075081,up_weak,0.695223,0.044057,0.218932,0.075081,up_p_lt_0.05,0.670074,0.047893,up_p_lt_0.05,-0.127151,2.961297e-02,-0.216229,7.883985e-02,down_p_lt_0.05
7,E10,R,67,0.895522,0.042142,0.745668,0.040370,0.745668,up_weak,0.187918,0.717480,0.040370,0.745668,up_weak,0.261180,0.515694,up_weak,-0.000820,7.578822e-01,-0.242220,4.828359e-02,flat
8,E11,I,67,0.820896,0.021071,0.897097,0.016102,0.897097,flat,0.529156,0.096487,0.016102,0.897097,up_weak,0.576619,0.056902,up_weak,-0.134685,2.985023e-02,-0.263672,3.108838e-02,down_p_lt_0.05
9,E11,M,67,0.865672,-0.187006,0.193942,-0.160689,0.193942,down_weak,-0.744972,0.189866,-0.160689,0.193942,down_weak,-0.823859,0.181947,down_weak,-0.119510,1.352667e-02,-0.233113,5.763717e-02,down_p_lt_0.05


First vs second half fatigue/learning proxy


,subject_id,finger_condition,mean_reaction_time_first_half,mean_reaction_time_second_half,n_trials_first_half,n_trials_second_half,success_rate_first_half,success_rate_second_half,success_rate_second_minus_first,fatigue_direction,reaction_time_second_minus_first
0,E1,I,2.088419,1.961172,32,32,0.906250,0.968750,0.062500,up_weak,-0.127246
1,E1,M,2.200159,1.805794,32,32,0.937500,0.875000,-0.062500,down_weak,-0.394365
2,E1,P,2.491971,2.393507,32,32,0.812500,0.906250,0.093750,up_weak,-0.098464
3,E1,R,1.952724,1.945272,32,32,0.937500,0.968750,0.031250,up_weak,-0.007452
4,E10,I,2.475518,2.455543,33,34,0.909091,0.823529,-0.085561,down_weak,-0.019975
5,E10,M,2.792674,2.150120,33,34,0.818182,0.970588,0.152406,up_weak,-0.642554
6,E10,P,2.464267,2.213386,33,34,0.878788,0.941176,0.062389,up_weak,-0.250881
7,E10,R,7.047507,2.304028,33,34,0.909091,0.882353,-0.026738,flat,-4.743479
8,E11,I,2.227676,2.010826,33,34,0.818182,0.823529,0.005348,flat,-0.216850
9,E11,M,2.413769,2.231409,33,34,0.909091,0.823529,-0.085561,down_weak,-0.182360


Between-participant duration/time statistics


,analysis,n_subjects,slope,linear_p_value,spearman_r,spearman_p,direction
0,between_subject_success_vs_mean_reaction_time,13,0.030218,0.645950,0.016506,0.957318,up_weak
1,between_subject_success_vs_median_reaction_time,13,-0.197718,0.182840,-0.057772,0.851296,down_weak
2,between_subject_success_vs_session_duration_min,13,-0.000073,0.953901,-0.228336,0.453061,flat
3,between_subject_success_vs_mean_abs_stiffness_...,13,0.003905,0.235305,0.390897,0.186607,flat


## 15. Finger identity across time and finger appearance order

This section addresses two additional time questions:

1. **Between fingers across time:** compare index/middle/ring/pinky success trajectories across normalized time within each finger's trials.
2. **Appearance of fingers:** compare the first, second, third, and fourth finger encountered in each participant session, independent of the anatomical finger identity.

Outputs include subject-level slopes, group time-bin curves by finger, the finger-order matrix for every participant, and paired contrasts between fingers/appearance positions.


In [ ]:
import importlib
pf = importlib.reload(pf)

finger_time_appearance = pf.compare_fingers_over_time_and_appearance(combined_success_trials)

finger_time_subject_summary = finger_time_appearance["subject_finger_summary"]
finger_time_group_bins = finger_time_appearance["group_finger_time_bins"]
finger_time_subject_bins = finger_time_appearance["subject_finger_time_bins"]
finger_appearance_order_summary = finger_time_appearance["appearance_order_summary"]
finger_by_appearance_order = finger_time_appearance["finger_by_appearance_order"]
finger_order_matrix = finger_time_appearance["finger_order_matrix"]
finger_time_slope_summary = finger_time_appearance["finger_slope_summary"]
stiffness_time_slope_summary = finger_time_appearance.get("stiffness_slope_summary", pd.DataFrame())
finger_time_slope_contrasts = finger_time_appearance["finger_slope_contrasts"]
finger_appearance_order_contrasts = finger_time_appearance["appearance_order_contrasts"]

pf.save_csv(finger_time_subject_summary, OUTPUT_ROOT, "finger_time_subject_summary.csv")
pf.save_csv(finger_time_group_bins, OUTPUT_ROOT, "finger_time_group_bins.csv")
pf.save_csv(finger_time_subject_bins, OUTPUT_ROOT, "finger_time_subject_bins.csv")
pf.save_csv(finger_appearance_order_summary, OUTPUT_ROOT, "finger_appearance_order_summary.csv")
pf.save_csv(finger_by_appearance_order, OUTPUT_ROOT, "finger_by_appearance_order.csv")
pf.save_csv(finger_order_matrix, OUTPUT_ROOT, "finger_order_matrix.csv")
pf.save_csv(finger_time_slope_summary, OUTPUT_ROOT, "finger_time_slope_summary.csv")
pf.save_csv(stiffness_time_slope_summary, OUTPUT_ROOT, "stiffness_time_slope_summary.csv")
pf.save_csv(finger_time_slope_contrasts, OUTPUT_ROOT, "finger_time_slope_contrasts.csv")
pf.save_csv(finger_appearance_order_contrasts, OUTPUT_ROOT, "finger_appearance_order_contrasts.csv")

print("Finger order matrix: which finger appeared 1st/2nd/3rd/4th for each participant")
display(finger_order_matrix)
print("Group success by finger and normalized within-finger time bin")
display(finger_time_group_bins)
print("Finger appearance-order summary")
display(finger_appearance_order_summary)
print("Finger time-slope summary")
display(finger_time_slope_summary)
print("Success slope by stiffness")
display(stiffness_time_slope_summary)
print("Paired contrasts between finger time slopes")
display(finger_time_slope_contrasts)
print("Paired contrasts between appearance-order success rates")
display(finger_appearance_order_contrasts)


## 16. Article-inspired figures adapted to this experiment: skin stretch and XY probing

The eLife article includes perception panels (psychometric curves, PSE, JND) and force/trajectory-style panels. For this experiment, the relevant substitutions are:

- **No grip force:** use the recorded skin-stretch gain/condition in **mm/m** and a skin-stretch command proxy, not grip force.
- **Probing from center in the XY plane:** use `object_x, object_y` from `tracking.csv` relative to screen center `(320, 240)`.
- **Article-style perception:** keep psychometric curves, PSE, JND, subject lines, and group average/dotted summary styling.

Important distinction: `motor_response_analisys_servo` is a separate hardware check/calibration for motor command response. This psychophysics notebook keeps the main analysis on the experiment variables that participants experienced: skin-stretch mm/m, XY probing from center, time, success, and finger/order effects.


In [ ]:
article_style_figure_paths, article_style_figure_manifest = pf.save_article_style_psychophysics_figures(
    OUTPUT_ROOT,
    combined_success_trials,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    finger_time_subject_summary,
    finger_appearance_order_summary,
    finger_by_appearance_order,
    preferred_subject=None,
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(article_style_figure_paths)} article-style psychophysics figures")
display(article_style_figure_manifest)


## 17. Plots and saved outputs

Saves PNG figures: subject ? finger curves, group curves, all-pooled curve, PSE/JND summaries, order effects, side bias, and QC plots. Subject-specific outputs are also written under `OUTPUT_ROOT/subjects/<subject_id>/`.

In [ ]:
figure_paths = pf.save_all_figures(
    OUTPUT_ROOT,
    combined_clean_trials,
    qc_summary,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    psychometric_input_group_by_finger,
    pse_jnd_group_by_finger,
    psychometric_input_group_all_pooled,
    pse_jnd_group_all_pooled,
    order_effects_binned,
    FIG_DPI,
)
time_fatigue_figure_paths = pf.save_time_fatigue_figures(
    OUTPUT_ROOT,
    success_by_reaction_time_bin,
    success_by_order_bin,
    fatigue_first_second_summary,
    success_summary_by_subject,
    success_trend_slopes_by_subject_finger,
    FIG_DPI,
)
finger_time_appearance_figure_paths = pf.save_finger_time_appearance_figures(
    OUTPUT_ROOT,
    finger_time_group_bins,
    finger_appearance_order_summary,
    finger_by_appearance_order,
    finger_time_slope_summary,
    FIG_DPI,
    stiffness_time_slope_summary=stiffness_time_slope_summary,
)
figure_paths = (
    figure_paths
    + time_fatigue_figure_paths
    + finger_time_appearance_figure_paths
    + article_style_figure_paths
)
print(f"Saved {len(figure_paths)} figure files.")
display(pd.DataFrame({"figure": [str(p) for p in figure_paths]}).head(60))

pf.save_subject_level_outputs(
    OUTPUT_ROOT,
    combined_clean_trials,
    combined_flagged_trials,
    pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled,
)

psychophysics_group_comparisons = pf.save_experiment_group_comparison_outputs(
    OUTPUT_ROOT,
    combined_success_trials,
    pse_jnd_by_subject_finger=pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled=pse_jnd_subject_pooled,
)
motor_control_method_references = pf.motor_control_method_references()
pf.save_csv(motor_control_method_references, OUTPUT_ROOT, "motor_control_method_references.csv")

print("N_E / L_E / L_P psychophysics group summary (transparent raw values are retained in raw_values_json and plotted faded in scope figures)")
display(psychophysics_group_comparisons["psychophysics_trial_group_metric_summary"].head(80))
print("Between-group psychophysics comparisons")
display(psychophysics_group_comparisons["psychophysics_trial_between_group_metric_comparisons"].head(80))
print("ANOVA-style factor statistics: one-way and two-way screening, subject-level collapsed where needed")
display(psychophysics_group_comparisons["psychophysics_factor_statistics"].head(120))
display(psychophysics_group_comparisons["psychophysics_factor_statistics_status"])
print("Motor-control / psychophysics method references saved for later citation")
display(motor_control_method_references[["citation_key", "year", "title", "doi", "analysis_use"]])

manifest = pf.analysis_manifest(OUTPUT_ROOT)
display(manifest)
print("Figures folder:", OUTPUT_ROOT / "figures")
print("Subject output folder:", OUTPUT_ROOT / "subjects")

## Interpretation notes

- `PSE` is the physical comparison value where `P(response_comparison_greater) = 0.5`.
- `JND = (x75 - x25) / 2`, in the same units as the stimulus values.
- `farajian_style_input_by_subject_finger.csv` mirrors the Farajian/Nisky perception scripts: stimulus difference, number of comparison-stiffer responses, and total repetitions.
- `correct_response` is the success variable: 1 means the physically stiffer stimulus was selected, 0 means it was not.
- `success_vs_order_slope > 0` suggests improvement/learning over the session; `< 0` suggests possible fatigue or decline. Treat this as a proxy because no direct subjective tiredness rating is present in `answers.csv`.
- `success_vs_reaction_time_slope` and `success_by_reaction_time_bin.csv` show whether longer answer duration is associated with better or worse success.
- `finger_time_group_bins.csv` compares success across normalized time within each finger, enabling direct index/middle/ring/pinky time-course comparison.
- `finger_appearance_order_summary.csv` compares the 1st/2nd/3rd/4th finger encountered in a participant session, which tests whether finger appearance/order itself matters.
- `finger_order_matrix.csv` records which anatomical finger appeared in each order position for every participant.
- Review `fit_warning`, `qc_warnings`, `combined_flagged_trials.csv`, and the per-subject trend tables before interpreting results.
- `motor_response_analisys_servo` is treated as a separate hardware calibration/check, not mixed into the main psychophysics variables.
- Rows excluded from fitting are preserved for audit; no raw data files are modified.

- All psychophysics group outputs now include the requested scopes when available: protocol/group (`N_E`, `L_E`, `L_P`, broad `N/L/P/E`, and all participants), participant, finger, stiffness/comparison value, success/failure, sex, age group, and workspace (`N` 40x50 cm; `L` 60x60 cm).
- `comparison_over_standard`, `signed_delta_over_standard`, and `abs_delta_over_standard` normalize analog stiffness values to the trial standard while preserving original values; workspace columns document the original physical space and do not alter stimulus units.
- Scope-summary plots use faded raw/background values plus the summary bar/point so every value remains transparent behind the group statistic.
- `psychophysics_factor_statistics.csv` is an ANOVA-style screening table. For binary repeated 2AFC trials, treat it as descriptive evidence; for manuscript-level inference prefer planned psychometric model comparisons or mixed/repeated-measures models with participant as a repeated/random factor.
- `motor_control_method_references.csv` lists the motor-control and haptic psychophysics articles used to guide the analysis additions and citation trail.
